# Khám phá dữ liệu: NYC Taxi Lakehouse (Trino)

Notebook này kết nối tới **Trino** để truy vấn trực tiếp vào **Nessie Catalog** (các bảng Iceberg). Việc này giúp phân tích các chỉ số KPIs một cách nhanh chóng với SQL chuẩn.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Kết nối tới Trino
engine = create_engine('trino://admin@trino:8080/nessie')
print("Đã kết nối thành công tới Trino!")

Đã kết nối thành công tới Trino!


## 1. Phân tích lượt đi và doanh thu theo Khu Vực (Top 10)

In [2]:
query_zone = '''
    SELECT 
        l.zone_name,
        SUM(f.total_trips) AS total_trips,
        SUM(f.total_revenue) AS total_revenue
    FROM nessie.gold.fact_revenue_by_zone f
    JOIN nessie.gold.dim_location l
      ON f.pulocation_id = l.location_id
    GROUP BY l.zone_name
    ORDER BY total_trips DESC
    LIMIT 10
'''
df_zone = pd.read_sql(query_zone, engine)
display(df_zone)

,zone_name,total_trips,total_revenue
0,JFK Airport,6231264,5.264553e+08
1,LaGuardia Airport,6036474,4.315200e+08
2,Midtown Center,4670305,1.824403e+08
3,Times Sq/Theatre District,4294364,1.889148e+08
4,East Village,4099820,1.274719e+08
5,Upper East Side South,3793160,1.158202e+08
6,East Chelsea,3776622,1.445126e+08
7,Union Sq,3584642,1.222780e+08
8,Midtown South,3434025,1.320878e+08
9,Upper East Side North,3415226,1.021769e+08


## 2. Xu hướng chuyến đi theo thời gian (Daily Trips Trend)

In [3]:
query_daily = '''
    SELECT 
        CAST(trip_date AS DATE) AS trip_date,
        trip_type,
        SUM(total_trips) AS total_trips
    FROM nessie.gold.fact_daily_trips
    GROUP BY CAST(trip_date AS DATE), trip_type
'''
df_daily = pd.read_sql(query_daily, engine)

# Biểu diễn dạng bảng pivot để dễ nhìn xu hướng các loại xe
df_pivot = df_daily.pivot(index='trip_date', columns='trip_type', values='total_trips').fillna(0)

# Tính cột tổng và sắp xếp giảm dần
df_pivot['total'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values('total', ascending=False)

# Thiết lập hiển thị tối đa 400 dòng để không bị cắt bớt bằng dấu ...
pd.set_option('display.max_rows', 400)
display(df_pivot)


trip_type,fhv,green,hvfhv,yellow,total
trip_date,,,,,
2024-12-14,5288,1913,882816,153091,1043108
2024-03-09,5540,1644,884336,129363,1020883
2024-12-13,9304,2051,843101,151898,1006354
2024-10-26,7538,1748,851236,142733,1003255
2024-03-23,5328,1332,875050,116919,998629
2024-12-07,5372,1562,848879,139932,995745
2024-03-02,5342,1541,869768,114741,991392
2024-09-28,10789,1498,833685,136820,982792
2024-11-22,11824,1946,823151,142434,979355


## 3. Thói Quen Thanh Toán (Payment Methods)

In [4]:
query_payment = '''
    SELECT 
        p.description AS payment_type,
        SUM(f.total_trips) AS total_trips
    FROM nessie.gold.fact_payment_summary f
    LEFT JOIN nessie.gold.dim_payment_type p
      ON f.payment_type_id = p.payment_type_id
    GROUP BY p.description
    ORDER BY total_trips DESC
'''
df_payment = pd.read_sql(query_payment, engine)
display(df_payment)

,payment_type,total_trips
0,Unknown,242901868
1,Credit Card,30298483
2,Cash,5388579
3,Flex Fare trip,3695906
4,Dispute,386371
5,No Charge,152574


## 4. Báo Cáo Tổng Hợp Tháng (Monthly Summary KPI)

In [5]:
query_monthly = '''
    SELECT 
        year, 
        month,
        SUM(total_trips) AS total_trips,
        SUM(total_revenue) AS total_revenue
    FROM nessie.gold.fact_monthly_summary
    GROUP BY year, month
    ORDER BY year DESC, month DESC
'''
df_monthly = pd.read_sql(query_monthly, engine)
display(df_monthly)

,year,month,total_trips,total_revenue
0,2024,12,24842391,8.186929e+08
1,2024,11,23828125,7.442254e+08
2,2024,10,23973239,7.717109e+08
3,2024,9,23099092,7.451060e+08
4,2024,8,22249901,6.938096e+08
5,2024,7,22528896,6.884425e+08
6,2024,6,23892077,7.615241e+08
7,2024,5,24603945,7.923234e+08
8,2024,4,23527636,7.215387e+08
9,2024,3,24997200,7.600095e+08
